# Setup Folders

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        !sudo apt-get install -y swig
        %pip install -q -r requirements.txt

In [2]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

remote_folder = None
if is_environment_colab_notebook():
    remote_folder = get_secret("worldModelsFolderPath")
    data_path = os.path.join(remote_folder, "data.tar")
    !cp "$data_path" .
    drive.flush_and_unmount()
    !rm -rf /root/.config/Google/DriveFS
    !rm -rf /content/drive
    !tar -xf data.tar
    !rm -rf data.tar
    drive.mount('/content/drive')

settings_path = "./settings.json"
data_folder = "./data"
output_folder = os.path.join(remote_folder, "weights/vae") if is_environment_colab_notebook() else "./weights/vae"
os.makedirs(output_folder, exist_ok=True)

local_data_folder = None

# Training Settings

In [3]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

2026-04-16 11:41:06 [INFO] Logger initialized.


In [4]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    IMAGE_CHANNELS = settings["vae"]["model"]["image_channels"]
    OBSERVATION_DIM = settings["vae"]["model"]["observation_dim"]
    HIDDEN_DIM = settings["vae"]["model"]["hidden_dim"]
    REPRESENTATION_DIM = settings["vae"]["model"]["representation_dim"]

    EPOCHS = settings["vae"]["train"]["epochs"]
    LR = settings["vae"]["train"]["learning_rate"]
    MAX_NORM = settings["vae"]["train"]["max_norm"]
    KLD_BETA = settings["vae"]["train"]["kld_beta"]
    TRAIN_RATIO = settings["vae"]["train"]["train_ratio"]
    EPOCHS_BETWEEN_TESTS = settings["vae"]["train"]["epochs_between_tests"]
    BATCH_SIZE = settings["vae"]["train"]["batch_size"]
    NUM_PRELOAD_FILES = settings["vae"]["train"]["num_preload_files"]
    NUM_DATASET_WORKERS = settings["vae"]["train"]["num_dataset_workers"]
    EARLY_STOPPING_TOLERANCE = settings["vae"]["train"]["early_stopping_tolerance"]
    EARLY_STOPPING_MIN_DELTA = settings["vae"]["train"]["early_stopping_min_delta"]
    TRAIN_IMAGE_LOG_INTERVAL = settings["vae"]["train"]["train_image_log_interval"]

WANDB_PROJECT = "world-models-paper"
WANDB_RUN_NAME = "vae"

def log_settings():
    logger.info(f"IMAGE_CHANNELS: {IMAGE_CHANNELS}")
    logger.info(f"OBSERVATION_DIM: {OBSERVATION_DIM}")
    logger.info(f"HIDDEN_DIM: {HIDDEN_DIM}")
    logger.info(f"REPRESENTATION_DIM: {REPRESENTATION_DIM}")
    logger.info("========================================")
    logger.info(f"EPOCHS: {EPOCHS}")
    logger.info(f"LR: {LR}")
    logger.info(f"MAX_NORM: {MAX_NORM}")
    logger.info(f"KLD_BETA: {KLD_BETA}")
    logger.info(f"TRAIN_RATIO: {TRAIN_RATIO}")
    logger.info(f"EPOCHS_BETWEEN_TESTS: {EPOCHS_BETWEEN_TESTS}")
    logger.info(f"BATCH_SIZE: {BATCH_SIZE}")
    logger.info(f"NUM_PRELOAD_FILES: {NUM_PRELOAD_FILES}")
    logger.info(f"NUM_DATASET_WORKERS: {NUM_DATASET_WORKERS}")
    logger.info(f"EARLY_STOPPING_TOLERANCE: {EARLY_STOPPING_TOLERANCE}")
    logger.info(f"EARLY_STOPPING_MIN_DELTA: {EARLY_STOPPING_MIN_DELTA}")
    logger.info(f"TRAIN_IMAGE_LOG_INTERVAL: {TRAIN_IMAGE_LOG_INTERVAL}")
    logger.info("========================================")
    logger.info(f"WANDB_PROJECT: {WANDB_PROJECT}")
    logger.info(f"WANDB_RUN_NAME: {WANDB_RUN_NAME}")

log_settings()

2026-04-16 11:41:07 [INFO] IMAGE_CHANNELS: 3
2026-04-16 11:41:07 [INFO] OBSERVATION_DIM: 64
2026-04-16 11:41:07 [INFO] HIDDEN_DIM: 1024
2026-04-16 11:41:07 [INFO] REPRESENTATION_DIM: 32
2026-04-16 11:41:07 [INFO] ========================================
2026-04-16 11:41:07 [INFO] EPOCHS: 500
2026-04-16 11:41:07 [INFO] LR: 0.001
2026-04-16 11:41:07 [INFO] MAX_NORM: 0.1
2026-04-16 11:41:07 [INFO] KLD_BETA: 4
2026-04-16 11:41:07 [INFO] TRAIN_RATIO: 0.8
2026-04-16 11:41:07 [INFO] EPOCHS_BETWEEN_TESTS: 5
2026-04-16 11:41:07 [INFO] BATCH_SIZE: 1024
2026-04-16 11:41:07 [INFO] NUM_PRELOAD_FILES: 10
2026-04-16 11:41:07 [INFO] NUM_DATASET_WORKERS: 8
2026-04-16 11:41:07 [INFO] EARLY_STOPPING_TOLERANCE: 5
2026-04-16 11:41:07 [INFO] EARLY_STOPPING_MIN_DELTA: 0.01
2026-04-16 11:41:07 [INFO] TRAIN_IMAGE_LOG_INTERVAL: 1000
2026-04-16 11:41:07 [INFO] ========================================
2026-04-16 11:41:07 [INFO] WANDB_PROJECT: world-models-paper
2026-04-16 11:41:07 [INFO] WANDB_RUN_NAME: vae


# Setup

In [5]:
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

from src.datasets.observations_dataset import ObservationsDataset
from src.models.vae import ConvVAE
from src.training.early_stopping import EarlyStopping
from src.training.vae_trainer import ConvVaeTrainer
from src.utils.torch import get_device

In [6]:
DEVICE = get_device(logger)

2026-04-16 11:41:14 [INFO] Using device: cuda:0


# Load Dataset

In [7]:
train_dataset, test_dataset = ObservationsDataset.train_test_split(data_folder,
                                                                   local_data_folder=local_data_folder,
                                                                   num_preloaded_files=NUM_PRELOAD_FILES,
                                                                   num_workers=NUM_DATASET_WORKERS,
                                                                   train_ratio=TRAIN_RATIO,
                                                                   shuffle_files=True,
                                                                   shuffle_file_samples=True,
                                                                   logger=logger)

e:\MyProjects\WorldModel\Demo1\world-models-implementation\src\datasets\observations_dataset.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path

In [8]:
train_size = len(train_dataset)
test_size = len(test_dataset)
logger.info(f"Train: {train_size}")
logger.info(f"Test: {test_size}")

2026-04-16 11:41:17 [INFO] Train: 66531
2026-04-16 11:41:17 [INFO] Test: 15429


In [9]:
example_observation = next(train_dataset)
logger.info(example_observation.shape)

2026-04-16 11:41:18 [INFO] torch.Size([3, 64, 64])


In [10]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [11]:
train_batches = len(train_dataloader)
test_batches = len(test_dataloader)
logger.info(f"Train batches: {train_batches}")
logger.info(f"Test batches: {test_batches}")

2026-04-16 11:41:20 [INFO] Train batches: 65
2026-04-16 11:41:20 [INFO] Test batches: 16


# Train

In [12]:
model = ConvVAE(image_channels=IMAGE_CHANNELS,
                h_dim=HIDDEN_DIM,
                z_dim=REPRESENTATION_DIM,
                device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
early_stopping = EarlyStopping(tolerance=EARLY_STOPPING_TOLERANCE, min_delta=EARLY_STOPPING_MIN_DELTA)

In [13]:
try:
    wandb_setup = {
        "api_key": get_secret('wandbApiKey'),
        "project": WANDB_PROJECT,
        "run_name": WANDB_RUN_NAME,
        "config": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
            "train_dataset_size": train_size,
            "test_dataset_size": test_size,
            "train_batches": train_batches,
            "test_batches": test_batches
        }
    }
except Exception as e:
    logger.error(f"Ignoring wandb logging, could not retrieve wandbApiKey secret. Error: {str(e)}")
    wandb_setup = None

In [20]:
trainer = ConvVaeTrainer(model=model,
                         weights_folder=output_folder,
                         train_dataloader=train_dataloader,
                         optimizer=optimizer,
                         num_epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         kld_beta=KLD_BETA,
                         train_image_log_interval=TRAIN_IMAGE_LOG_INTERVAL,
                         load_checkpoint=True,
                         max_norm=MAX_NORM,
                         device=DEVICE,
                         test_dataloader=test_dataloader,
                         epochs_between_tests=EPOCHS_BETWEEN_TESTS,
                         early_stopper=early_stopping,
                         wandb_setup=wandb_setup,
                         logger=logger)

2026-04-16 11:39:52 [INFO] No existing checkpoints found. Starting from scratch.


ValueError: Mandatory parameter api_key is None

In [ ]:
# Repeat logging of important information so it is available on wandb run
log_settings()
_ = get_device(logger)

In [ ]:
model = trainer.train()

# Testing

In [ ]:
def test_observation(model, observation):
    model.eval()
    with torch.no_grad():
        z, _, _ = model.encode(observation.unsqueeze(0))
        decoded = model.decode(z)
    _, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(observation.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title('Original Observation')
    axes[0].axis('off')
    axes[1].imshow(decoded.squeeze(0).permute(1, 2, 0).cpu().numpy())
    axes[1].set_title('Decoded Observation')
    axes[1].axis('off')
    plt.show()

In [ ]:
i = 0
for example_observation in iter(test_dataset):
    test_observation(model, example_observation.to(DEVICE))
    i += 1
    if i == 10:
        break